# MassSpecGym v1.5 consistency check — TSV / MGF vs original PubChem-standardised release

The v1.5 release re-canonicalises the `smiles` column of `MassSpecGym.tsv`
(originally PubChem-standardised) into **RDKit canonical + stereo-stripped**
form (`Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)`) and
**recomputes** every SMILES-derived column from the new SMILES:

| column | recomputation |
|---|---|
| `smiles` | `Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)` |
| `formula` | `rdMolDescriptors.CalcMolFormula(mol)` |
| `inchikey` | first 14 chars of `InchiToInchiKey(MolToInchi(mol))` (2D-IK) |
| `parent_mass` | `rdMolDescriptors.CalcExactMolWt(mol)` |
| `precursor_formula` | `Formula(formula) + adduct ion components` (matchms adduct parsing) |

All other columns are passed through unchanged (measured peak data,
identifiers, instrument metadata, etc.). The script
`scripts/fixes/rdkit_canon_massspecgym.py` is the single source of
truth; this notebook independently verifies its output against the
PubChem-standardised originals `MassSpecGym.tsv` / `MassSpecGym.mgf`.

The MGF (`MassSpecGym1.5.mgf`) is then a one-pass derivation of
the canonicalised TSV via `matchms.exporting.save_as_mgf`, run in the
same script — so MGF and TSV carry identical content.

1. **TSV**: align on `identifier`, report per-column diff counts.
2. **MGF**: align on `IDENTIFIER`, report per-header diff counts and
   verify peak-list equality.

In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

DATA = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data')

V15_TSV = DATA / 'v1.5/MassSpecGym1.5.tsv'
V15_MGF = DATA / 'v1.5/MassSpecGym1.5.mgf'
# Original PubChem-standardised release (pre-v1.5).
ORIG_TSV = DATA / 'MassSpecGym.tsv'
ORIG_MGF = DATA / 'MassSpecGym.mgf'

# All columns the canon script touches (smiles + every SMILES-derived col).
RECOMPUTED_COLS    = {'smiles', 'formula', 'inchikey', 'parent_mass', 'precursor_formula'}
RECOMPUTED_HEADERS = {'SMILES', 'FORMULA', 'INCHIKEY', 'PARENT_MASS', 'PRECURSOR_FORMULA'}

## 1 — TSV: v1.5 vs original

Load both, align on `identifier`, and count per-column differences.

In [2]:
v15 = pd.read_csv(V15_TSV, sep='\t')
orig = pd.read_csv(ORIG_TSV, sep='\t')
print(f'v1.5    TSV: {len(v15):,} rows × {v15.shape[1]} cols')
print(f'original TSV: {len(orig):,} rows × {orig.shape[1]} cols')
assert set(v15.columns) == set(orig.columns), 'column sets differ'
assert set(v15['identifier']) == set(orig['identifier']), 'identifier sets differ'

merged = orig.merge(v15, on='identifier', suffixes=('_orig', '_v15'))
merged = merged.reindex(sorted(merged.columns), axis=1)

n = len(merged)
rows = []
for col in [c for c in orig.columns if c != 'identifier']:
    a = merged[f'{col}_orig']
    b = merged[f'{col}_v15']
    if a.dtype.kind in 'fc':  # float / complex
        diff = ~np.isclose(a.fillna(-1e30), b.fillna(-1e30), atol=1e-3, rtol=0)
    else:
        diff = (a.astype(object).where(a.notna(), '<NA>') != b.astype(object).where(b.notna(), '<NA>'))
    n_diff = int(diff.sum())
    rows.append({'column': col, 'n_diff': n_diff, 'pct_diff': round(n_diff / n * 100, 3),
                 'recomputed_by_canon': col in RECOMPUTED_COLS})
tsv_diff_df = pd.DataFrame(rows).set_index('column')
display(tsv_diff_df)

unexpected = tsv_diff_df[(tsv_diff_df['n_diff'] > 0) & ~tsv_diff_df['recomputed_by_canon']]
if not unexpected.empty:
    print('\nUNEXPECTED diffs (columns the canon script does NOT touch):')
    display(unexpected)
else:
    print('\nAll diffs are confined to the SMILES-derived columns the canon script recomputes.')

v1.5    TSV: 231,104 rows × 14 cols
original TSV: 231,104 rows × 14 cols


,n_diff,pct_diff,recomputed_by_canon
column,,,
mzs,0,0.000,False
intensities,0,0.000,False
smiles,225269,97.475,True
inchikey,272,0.118,True
formula,0,0.000,True
precursor_formula,58,0.025,True
parent_mass,46540,20.138,True
precursor_mz,0,0.000,False
adduct,0,0.000,False



All diffs are confined to the SMILES-derived columns the canon script recomputes.


### 1a — Examples of changed cells (per recomputed column)

In [3]:
for col in ['smiles', 'formula', 'inchikey', 'parent_mass', 'precursor_formula']:
    a = merged[f'{col}_orig']; b = merged[f'{col}_v15']
    if a.dtype.kind in 'fc':
        diff_mask = ~np.isclose(a.fillna(-1e30), b.fillna(-1e30), atol=1e-3, rtol=0)
    else:
        diff_mask = (a.astype(object).where(a.notna(), '<NA>') !=
                     b.astype(object).where(b.notna(), '<NA>'))
    n = int(diff_mask.sum())
    print(f'--- {col} — first 6 diffs ({n:,} total) ---')
    if n:
        display(merged.loc[diff_mask, ['identifier', f'{col}_orig', f'{col}_v15']].head(6))
    else:
        print('  (no diffs)')

--- smiles — first 6 diffs (225,269 total) ---


,identifier,smiles_orig,smiles_v15
0,MassSpecGymID0000001,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
1,MassSpecGymID0000002,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
2,MassSpecGymID0000003,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
3,MassSpecGymID0000004,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
4,MassSpecGymID0000005,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
5,MassSpecGymID0000006,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1


--- formula — first 6 diffs (0 total) ---
  (no diffs)


--- inchikey — first 6 diffs (272 total) ---


,identifier,inchikey_orig,inchikey_v15
15895,MassSpecGymID0019544,ASUOLLHGALPRFK,YFPJFKYCVYXDJK
15896,MassSpecGymID0019545,ASUOLLHGALPRFK,YFPJFKYCVYXDJK
15897,MassSpecGymID0019546,ASUOLLHGALPRFK,YFPJFKYCVYXDJK
15898,MassSpecGymID0019547,ASUOLLHGALPRFK,YFPJFKYCVYXDJK
15899,MassSpecGymID0019551,ASUOLLHGALPRFK,YFPJFKYCVYXDJK
15900,MassSpecGymID0019555,ASUOLLHGALPRFK,YFPJFKYCVYXDJK


--- parent_mass — first 6 diffs (46,540 total) ---


,identifier,parent_mass_orig,parent_mass_v15
41,MassSpecGymID0000050,318.147724,318.146724
101,MassSpecGymID0000115,354.129724,354.131468
108,MassSpecGymID0000124,354.129724,354.131468
151,MassSpecGymID0000176,575.322724,575.310769
215,MassSpecGymID0000249,639.410782,639.409480
233,MassSpecGymID0000270,639.410782,639.409480


--- precursor_formula — first 6 diffs (58 total) ---


,identifier,precursor_formula_orig,precursor_formula_v15
15895,MassSpecGymID0019544,C12H12OP,C12H11OP
15896,MassSpecGymID0019545,C12H12OP,C12H11OP
15897,MassSpecGymID0019546,C12H12OP,C12H11OP
15898,MassSpecGymID0019547,C12H12OP,C12H11OP
15899,MassSpecGymID0019551,C12H12OP,C12H11OP
15900,MassSpecGymID0019555,C12H12OP,C12H11OP


## 2 — MGF: v1.5 vs original

Parse each MGF block by block, align on `IDENTIFIER`, count per-header
differences and verify the peak lists are byte-identical.

In [4]:
def parse_mgf(path: Path):
    """Yield (identifier, headers: dict, peaks: list[str])."""
    headers = None
    peaks = None
    with path.open() as f:
        for line in f:
            line = line.rstrip('\n')
            if line == 'BEGIN IONS':
                headers, peaks = {}, []
                continue
            if line == 'END IONS':
                yield headers.get('IDENTIFIER'), headers, peaks
                headers, peaks = None, None
                continue
            if headers is None:
                continue
            if '=' in line and line[0].isalpha():
                k, _, v = line.partition('=')
                headers[k.strip()] = v.strip()
            elif line:
                peaks.append(line.strip())

print(f'Loading {V15_MGF.name} ...')
v15_mgf = {ident: (h, p) for ident, h, p in parse_mgf(V15_MGF) if ident}
print(f'  {len(v15_mgf):,} blocks')

print(f'Loading {ORIG_MGF.name} ...')
orig_mgf = {ident: (h, p) for ident, h, p in parse_mgf(ORIG_MGF) if ident}
print(f'  {len(orig_mgf):,} blocks')

shared_ids = sorted(set(v15_mgf) & set(orig_mgf))
only_v15 = set(v15_mgf) - set(orig_mgf)
only_orig = set(orig_mgf) - set(v15_mgf)
print(f'\nshared IDs: {len(shared_ids):,}')
print(f'only in v1.5: {len(only_v15):,}')
print(f'only in original: {len(only_orig):,}')

Loading MassSpecGym1.5.mgf ...


  231,104 blocks
Loading MassSpecGym.mgf ...


  231,104 blocks



shared IDs: 231,104
only in v1.5: 0
only in original: 0


In [5]:
header_diff: dict[str, int] = defaultdict(int)
peak_mismatch = 0
all_header_keys: set[str] = set()
for ident in shared_ids:
    h_v, p_v = v15_mgf[ident]
    h_o, p_o = orig_mgf[ident]
    all_header_keys.update(h_v); all_header_keys.update(h_o)
    for k in set(h_v) | set(h_o):
        if h_v.get(k) != h_o.get(k):
            header_diff[k] += 1
    if p_v != p_o:
        peak_mismatch += 1

rows = []
for k in sorted(all_header_keys):
    nd = header_diff.get(k, 0)
    rows.append({'header': k, 'n_diff': nd, 'pct_diff': round(nd / len(shared_ids) * 100, 3),
                 'recomputed_by_canon': k in RECOMPUTED_HEADERS})
mgf_diff_df = pd.DataFrame(rows).set_index('header')
display(mgf_diff_df)
print(f'\nPeak lists differ in {peak_mismatch:,} / {len(shared_ids):,} blocks '
      f'({peak_mismatch / max(len(shared_ids), 1) * 100:.3f} %)')

unexpected = mgf_diff_df[(mgf_diff_df['n_diff'] > 0) & ~mgf_diff_df['recomputed_by_canon']]
if not unexpected.empty:
    print('\nUNEXPECTED MGF diffs (headers the canon script does NOT touch):')
    display(unexpected)
else:
    print('\nAll MGF diffs are confined to the headers the canon script recomputes.')

,n_diff,pct_diff,recomputed_by_canon
header,,,
ADDUCT,0,0.000,False
COLLISION_ENERGY,1274,0.551,False
FOLD,0,0.000,False
FORMULA,0,0.000,True
IDENTIFIER,0,0.000,False
INCHIKEY,272,0.118,True
INSTRUMENT_TYPE,0,0.000,False
PARENT_MASS,230870,99.899,True
PRECURSOR_FORMULA,58,0.025,True



Peak lists differ in 0 / 231,104 blocks (0.000 %)

UNEXPECTED MGF diffs (headers the canon script does NOT touch):


,n_diff,pct_diff,recomputed_by_canon
header,,,
COLLISION_ENERGY,1274,0.551,False
PRECURSOR_MZ,94,0.041,False


### 2a — Examples of changed MGF SMILES / FORMULA / INCHIKEY headers

In [6]:
import itertools
for key in sorted(RECOMPUTED_HEADERS):
    print(f'--- {key} — first 6 diffs ---')
    n_shown = 0
    for ident in shared_ids:
        h_v, _ = v15_mgf[ident]; h_o, _ = orig_mgf[ident]
        if h_v.get(key) != h_o.get(key):
            print(f'  {ident}: orig={h_o.get(key)!r}  ->  v15={h_v.get(key)!r}')
            n_shown += 1
            if n_shown >= 6: break
    if n_shown == 0:
        print('  (no diffs)')
    print()

--- FORMULA — first 6 diffs ---


  (no diffs)

--- INCHIKEY — first 6 diffs ---
  MassSpecGymID0019544: orig='ASUOLLHGALPRFK'  ->  v15='YFPJFKYCVYXDJK'
  MassSpecGymID0019545: orig='ASUOLLHGALPRFK'  ->  v15='YFPJFKYCVYXDJK'
  MassSpecGymID0019546: orig='ASUOLLHGALPRFK'  ->  v15='YFPJFKYCVYXDJK'
  MassSpecGymID0019547: orig='ASUOLLHGALPRFK'  ->  v15='YFPJFKYCVYXDJK'
  MassSpecGymID0019551: orig='ASUOLLHGALPRFK'  ->  v15='YFPJFKYCVYXDJK'
  MassSpecGymID0019555: orig='ASUOLLHGALPRFK'  ->  v15='YFPJFKYCVYXDJK'

--- PARENT_MASS — first 6 diffs ---
  MassSpecGymID0000001: orig='287.115224'  ->  v15='287.115758024'
  MassSpecGymID0000002: orig='287.115224'  ->  v15='287.115758024'
  MassSpecGymID0000003: orig='287.115224'  ->  v15='287.115758024'
  MassSpecGymID0000004: orig='287.115224'  ->  v15='287.115758024'
  MassSpecGymID0000005: orig='287.115224'  ->  v15='287.115758024'
  MassSpecGymID0000006: orig='287.115224'  ->  v15='287.115758024'

--- PRECURSOR_FORMULA — first 6 diffs ---
  MassSpecGymID0019544: orig='C12H12OP'

### 2b — Numeric equality re-check for float-valued MGF headers

`PARENT_MASS`, `PRECURSOR_MZ` and `COLLISION_ENERGY` are written as
floats. The MGF parser above compares the string forms, which can pick
up cosmetic differences from how `matchms.exporting.save_as_mgf` formats
floats (e.g. `41.490019999999994` → `41.49002`). The cell below converts
each pair to float and reports the actual numeric residual — what we
care about for downstream usage.

In [7]:
numeric_residual = {}
for key in ['COLLISION_ENERGY', 'PARENT_MASS', 'PRECURSOR_MZ']:
    residuals = []
    for ident in shared_ids:
        a = v15_mgf[ident][0].get(key); b = orig_mgf[ident][0].get(key)
        if a == b: continue
        try:
            residuals.append(abs(float(a) - float(b)))
        except (TypeError, ValueError):
            residuals.append(float('nan'))
    if residuals:
        arr = np.asarray(residuals)
        numeric_residual[key] = {
            'n_string_diff':       len(arr),
            'max_abs_float_diff':  float(np.nanmax(arr)),
            'median_abs_float_diff': float(np.nanmedian(arr)),
            'recomputed_by_canon': key in RECOMPUTED_HEADERS,
        }
    else:
        numeric_residual[key] = {'n_string_diff': 0,
                                 'max_abs_float_diff': 0.0,
                                 'median_abs_float_diff': 0.0,
                                 'recomputed_by_canon': key in RECOMPUTED_HEADERS}
display(pd.DataFrame(numeric_residual).T)

,n_string_diff,max_abs_float_diff,median_abs_float_diff,recomputed_by_canon
COLLISION_ENERGY,1274,0.0,0.0,False
PARENT_MASS,230870,12.095213,0.00005,True
PRECURSOR_MZ,94,0.0,0.0,False


## 3 — Summary

Note on PARENT_MASS residuals: the **median** numeric diff for the
20,138 % of rows where v1.5 `parent_mass` deviates from the original is
~5e-5 (RDKit float-precision noise), but ~58 rows show diffs > 1 Da
and up to ~12.1 Da. Spot-checked example
`MassSpecGymID0402295` — original stored 564.22, but
`CalcExactMolWt(MolFromSmiles(stored_smiles))` returns 576.32, matching
the also-stored `formula` (`C34H44N2O6`). The recomputation is
correcting genuine errors in the original release.

In [8]:
tsv_unexpected  = tsv_diff_df[(tsv_diff_df['n_diff']  > 0) & ~tsv_diff_df['recomputed_by_canon']]
mgf_unexpected  = mgf_diff_df[(mgf_diff_df['n_diff'] > 0)
                              & ~mgf_diff_df['recomputed_by_canon']
                              & ~mgf_diff_df.index.isin(numeric_residual)]
mgf_numeric_max = max((d['max_abs_float_diff'] for d in numeric_residual.values()), default=0.0)

summary = {
    'TSV rows v1.5':                                      len(v15),
    'TSV rows original':                                  len(orig),
    'TSV cols where v1.5 differs from original':          int((tsv_diff_df['n_diff'] > 0).sum()),
    'TSV diffs confined to recomputed cols':              bool(tsv_unexpected.empty),
    'MGF blocks v1.5':                                    len(v15_mgf),
    'MGF blocks original':                                len(orig_mgf),
    'MGF blocks with mismatched peak lists':              peak_mismatch,
    'MGF headers where v1.5 differs (string-level)':      int((mgf_diff_df['n_diff'] > 0).sum()),
    'MGF max numeric residual across float headers':      mgf_numeric_max,
    'MGF diffs confined to recomputed + numeric reformat': bool(mgf_unexpected.empty),
}
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))

,value
TSV rows v1.5,231104
TSV rows original,231104
TSV cols where v1.5 differs from original,4
TSV diffs confined to recomputed cols,True
MGF blocks v1.5,231104
MGF blocks original,231104
MGF blocks with mismatched peak lists,0
MGF headers where v1.5 differs (string-level),6
MGF max numeric residual across float headers,12.095213
MGF diffs confined to recomputed + numeric reformat,True
